In [ ]:
from datetime import datetime
from lightningrod.utils import config
from dotenv import load_dotenv
from lightningrod import (
    LightningRod,
    GdeltSeedGenerator,
    ForwardLookingQuestionGenerator,
    QuestionPipeline,
    QuestionRenderer,
    WebSearchLabeler,
    MultipleChoiceAnswerType,
    multiple_choice_example,
)

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

In [ ]:
instructions = """
Generate multiple-choice forecasting questions about a specific future real-world event, based on recent news coverage.

Each question MUST resolve within the next three months.
"""

good_examples = [
    # 3 options
    multiple_choice_example(
        "What will the European Central Bank decide at its April 10, 2025 monetary policy meeting regarding its benchmark interest rate?",
        ["Rate increase", "No change", "Rate cut"],
        label=1,
    ),
    # 4 options
    multiple_choice_example(
        "By July 15, 2025, what will be the status of the proposed merger between Kroger and Albertsons?",
        ["Fully approved", "Regulator blocked", "Deal withdrawn", "Under review"],
        label=1,
    ),
    # 5 options
    multiple_choice_example(
        "By June 30, 2025, how many countries will have formally ratified the Global Plastics Treaty adopted in November 2024?",
        ["Fewer than 20", "20\u201339", "40\u201359", "60\u201379", "80 or more"],
        label=1,
    ),
    # 6 options
    multiple_choice_example(
        "By August 31, 2025, what stage will the United Kingdom's Sizewell C nuclear power project have reached?",
        ["Pre-construction", "Investment approved", "Site preparation", "Reactor construction", "Power generation", "Project canceled"],
        label=1,
    ),
]

bad_examples = [
    multiple_choice_example(
        "Which of the following will occur by December 31, 2025?",
        ["Japan raises interest rates", "Apple releases a foldable iPhone", "Brazil hosts a climate summit", "None of the above"],
        comment="Multiple unrelated events; violates single-event and IIA criteria.",
    ),
    multiple_choice_example(
        "What will Russia do by July 1, 2025 regarding Ukraine?",
        ["Launch a new offensive", "Launch a new offensive and mobilize additional troops", "Take no new military action"],
        comment="option_1 is a subset of option_0; logical nesting breaks IIA.",
    ),
    multiple_choice_example(
        "Will the merger between Company X and Company Y be approved by regulators by May 1, 2025?",
        ["Yes", "No", "Still under review", "Not reported"],
        comment="Binary yes/no disguised as multiple choice; mixes outcome with reporting status.",
    ),
    multiple_choice_example(
        "By October 2025, how will inflation in Argentina change?",
        ["Increase significantly", "Increase slightly", "Stay about the same", "Decrease"],
        comment="Vague, non-verifiable magnitude; multiple answers could be correct.",
    ),
    multiple_choice_example(
        "Which of the following will occur by January 31, 2025?",
        ["A major new sanctions package is announced", "A ceasefire agreement is signed", "Both option_0 and option_1", "Neither option_0 nor option_1"],
        comment="Combines independent events; explicit IIA violation.",
    ),
    multiple_choice_example(
        "Who will win the presidential election in the United States in 2024?",
        ["Donald Trump", "Kamala Harris"],
        comment="Only two options; use binary answer type instead.",
    ),
]

In [ ]:
answer_type = MultipleChoiceAnswerType()

pipeline = QuestionPipeline(
    seed_generator=GdeltSeedGenerator(
        start_date=datetime(2024, 7, 1),
        end_date=datetime(2025, 11, 30),
        articles_per_interval=35,
        interval_duration_days=7,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        questions_per_seed=2,
        answer_type=answer_type,
    ),
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.9,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

dataset = lr.transforms.run(pipeline, max_questions=200)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $4.43                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons  ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ GdeltSeedGenerato… │ Complete             │   2 │  60 │        0 │      0 │ -                  │      33s │  │
│  │ ForwardLookingQue… │ Complete             │  60 │ 107 │       13 │      0 │ date_close not     │       7s │  │
│  │                    │                      │     │     │          │        │ after event_date   │          │  │
│  │                    │                      │     │     │          │        │ (13)               │          │  │
│  │ WebSearchLabelerT… │ Complete             │ 107 │ 104 │        3 │      0 │ Undetermined label │      52s │  │
│  │                    │                      │     │     │          │        │ (2), Resolution    │          │  │
│  │                    │                      │     │     │          │        │ date is before     │          │  │
│  │                    │                      │     │     │          │        │ seed creation date │          │  │
│  │                    │                      │     │     │          │        │ (1)                │          │  │
│  │ QuestionRendererT… │ Complete             │ 104 │ 104 │        0 │      0 │ -                  │       0s │  │
│  └────────────────────┴──────────────────────┴─────┴─────┴──────────┴────────┴────────────────────┴──────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
import pandas as pd

pd.DataFrame(dataset.flattened())

,sample_id,is_valid,question_text,date_close,event_date,resolution_criteria,prediction_date,answer_sources,seed_text,seed_url,...,meta_sample_id,invalid_reason,meta_parent_sample_id,meta_processing_time_ms,label,answer_type,label_confidence,resolution_date,reasoning,prompt
0,0872f5a4-8490-4b7b-9a9e-85d59f0d0a35,False,"By October 1, 2025, will the governments of Ta...",2025-10-01T00:00:00,2025-10-01T00:00:00,This question resolves based on official gover...,2025-10-01T00:00:00,None,"Taiwan ""will not agree"" to making 50 percent o...",https://economictimes.indiatimes.com/news/inte...,...,b47834d7-9ed9-4a97-bdfb-1ec431be3ac1,date_close not after event_date,7101831d-1c33-413d-b526-a93167e19299,3.948,NaN,NaN,NaN,NaN,NaN,NaN
1,0a37abee-26e0-4548-92e0-6fbb071eb472,True,"By October 13, 2025, how many of the remaining...",2025-10-13T00:00:00,2025-10-04T00:00:00,This question will be resolved by official rep...,2025-10-04T00:00:00,https://vertexaisearch.cloud.google.com/ground...,حماس یرغمالیوں کی رہائی کے لیے رضا مند\n4 اکتو...,https://www.dw.com/ur/%D8%AD%D9%85%D8%A7%D8%B3...,...,d3576122-cebd-4c1f-9953-eda6f71e6645,NaN,15fa7c71-e011-4201-921d-248c96ffe144,115340.409,None,multiple_choice,0.95,2025-10-13T00:00:00,The resolution criteria specify that the count...,"QUESTION:\nBy October 13, 2025, how many of th..."
2,0b166158-f96a-4230-bd3c-d07a553b9d5b,True,"By November 1, 2025, will the Israeli governme...",2025-11-01T00:00:00,2025-09-30T00:00:00,The question will be resolved based on officia...,2025-09-30T00:00:00,https://vertexaisearch.cloud.google.com/ground...,Yihad islámica rechaza plan de Trump para Gaza...,https://www.dw.com/es/yihad-isl%C3%A1mica-rech...,...,684a1d77-b229-472b-b926-9022e5bfa239,NaN,641139e0-b8b9-4bd4-be8e-d2b36c98c53f,44088.980,Agreement signed by both parties,multiple_choice,1.00,2025-10-09T00:00:00,"The close date for the question is 2025-11-01,...","QUESTION:\nBy November 1, 2025, will the Israe..."
3,0c726a5d-4631-4292-9d39-36d065197533,True,What will be the status of the Puducherry gove...,2025-02-01T00:00:00,2024-11-07T00:00:00,The question will be resolved by checking offi...,2024-11-07T00:00:00,https://vertexaisearch.cloud.google.com/ground...,The CPI (M) has urged the government to initia...,https://www.thehindu.com/news/cities/puducherr...,...,8cc6a610-0839-4b0d-95dc-21b6782de175,NaN,6889955a-ae5c-4e6c-a506-38bfbbf3b117,85866.146,Union Government formally approves additional ...,multiple_choice,0.90,2025-02-01T00:00:00,"The Puducherry government, led by Chief Minist...",QUESTION:\nWhat will be the status of the Pudu...
4,0d7ea1a0-6087-4371-9f3d-78e6c439880e,True,"By December 11, 2024, will any U.S. state have...",2024-12-12T00:00:00,2024-11-07T00:00:00,"Under the Electoral Count Reform Act, December...",2024-11-07T00:00:00,https://vertexaisearch.cloud.google.com/ground...,Washington: US President Joe Biden on Thursday...,https://economictimes.indiatimes.com/news/inte...,...,9c80f73f-8e3d-47c3-8cf0-f449e81499df,NaN,080fb9da-f406-438b-8ea0-945f5bc20c69,144343.709,One state has failed to issue a certificate,multiple_choice,1.00,2024-12-11T00:00:00,The close date is 2024-12-12 and the question ...,"QUESTION:\nBy December 11, 2024, will any U.S...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,f609e214-4219-40dd-a4a0-242a14948e5b,True,"By October 15, 2025, how many of the remaining...",2025-10-15T00:00:00,2025-10-04T00:00:00,This question will be resolved using official ...,2025-10-04T00:00:00,https://vertexaisearch.cloud.google.com/ground...,Israel's army said Saturday that it would adva...,https://economictimes.indiatimes.com/news/inte...,...,6ee22816-e4f5-4a79-969a-27f1ba52c569,NaN,8c6d31b6-e45f-44f1-8619-9f99b02c4bcf,125137.335,21 to 47 hostages,multiple_choice,1.00,2025-10-15T00:00:00,"The close date: 2025-10-15, the question date:...","QUESTION:\nBy October 15, 2025, how many of th..."
116,fb231be7-7dc7-4c13-a77d-acb7be185360,True,"By October 20, 2025, 